# FEA sequence-order ablation: significance tests

This notebook contains two separate paired comparisons: **Ordered vs. Shuffled** (Section 3) and **Ordered vs. Mean** (Section 4). Section 5 applies Holm correction to these two primary p-values within the FEA modality. Run all cells from top to bottom with `fea_test_predictions.csv` beside this notebook.

The statistical procedures, difference direction (Ordered minus alternative), random seed (42), 1,000,000 bootstrap resamples, batch size (10,000), and alpha (0.05) are unchanged from the separate notebooks. Each comparison starts its bootstrap generator with the same original seed.

FEA inference uses 378 paired reenactments. The primary test is exact two-sided McNemar; a paired bootstrap supplies the accuracy-difference confidence interval.

The one-sample t-tests remain sensitivity checks, and participant summaries remain descriptive. The 95% confidence intervals are unadjusted marginal intervals. Dependence between different reenactments of the same participant is not modeled.

## 1. Shared setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import ttest_1samp
from statsmodels.stats.contingency_tables import mcnemar


PREDICTIONS_PATH = Path("fea_test_predictions.csv")

RANDOM_SEED = 42

# Same bootstrap settings as the existing Static-vs.-Dynamic FEA analysis.
N_BOOTSTRAP = 1_000_000
BOOTSTRAP_BATCH_SIZE = 10_000

ALPHA = 0.05

from statsmodels.stats.multitest import multipletests
results = {}

## 2. Load and validate predictions

Load the common prediction table once. Both comparisons use the same samples, labels, and participant assignments.

In [2]:
prediction_df = pd.read_csv(PREDICTIONS_PATH)

print(f"Rows:    {len(prediction_df)}")
print(f"Columns: {len(prediction_df.columns)}")

prediction_df.head()

Rows:    378
Columns: 35


,reenactment_id,timestamp,set_id,participant_id,level_id,emoji_id,true_label_id,true_label,ordered_pred_id,ordered_pred,...,shuffled_prob_surprise,mean_pred_id,mean_pred,mean_prob_anger,mean_prob_disgust,mean_prob_fear,mean_prob_happiness,mean_prob_neutral,mean_prob_sadness,mean_prob_surprise
0,1700478995850-2-1-1-0-0,1700478995850,2,1,1,0,0,Anger,0,Anger,...,0.003162,0,Anger,0.838640,0.034060,0.010857,6.689494e-04,0.001781,0.086494,0.027500
1,1700478998549-2-1-1-1-5,1700478998549,2,1,1,1,5,Sadness,5,Sadness,...,0.000900,5,Sadness,0.000826,0.000346,0.000051,5.587236e-07,0.000006,0.998732,0.000038
2,1700479001137-2-1-1-2-3,1700479001137,2,1,1,2,3,Happiness,3,Happiness,...,0.040769,3,Happiness,0.000762,0.017813,0.249371,5.898122e-01,0.000365,0.009472,0.132405
3,1700479004312-2-1-1-3-0,1700479004312,2,1,1,3,0,Anger,4,Neutral,...,0.010360,4,Neutral,0.096489,0.024780,0.020434,1.909493e-02,0.783427,0.018887,0.036888
4,1700479005401-2-1-1-4-0,1700479005401,2,1,1,4,0,Anger,0,Anger,...,0.003727,0,Anger,0.815663,0.054917,0.007815,1.686807e-03,0.008238,0.085832,0.025849


In [3]:
required_columns = {
    "reenactment_id",
    "participant_id",
    "true_label_id",
    "ordered_pred_id",
    "shuffled_pred_id"
}

missing_columns = required_columns - set(prediction_df.columns)
assert not missing_columns, f"Missing required columns: {sorted(missing_columns)}"

assert len(prediction_df) == 378
assert prediction_df["reenactment_id"].is_unique
assert prediction_df["participant_id"].nunique() == 8
assert prediction_df["true_label_id"].between(0, 6).all()
assert prediction_df["ordered_pred_id"].between(0, 6).all()
assert prediction_df["shuffled_pred_id"].between(0, 6).all()
assert prediction_df.groupby("true_label_id").size().eq(54).all()

print("Prediction table validated.")

# Both alternatives are required for the within-modality Holm family.
for condition in ["ordered", "shuffled", "mean"]:
    column = f"{condition}_pred_id"
    assert column in prediction_df.columns
    assert prediction_df[column].notna().all()
    assert prediction_df[column].isin(range(7)).all()
assert prediction_df["true_label_id"].isin(range(7)).all()

Prediction table validated.


### Shared bootstrap function

This is the unchanged resampling function from the separate notebooks; it is called once for each comparison below.

In [4]:
def paired_bootstrap_ci(
    differences: np.ndarray,
    n_bootstrap: int,
    batch_size: int,
    alpha: float,
    random_seed: int
) -> tuple[float, float]:
    rng = np.random.default_rng(random_seed)
    n = len(differences)

    bootstrap_means = np.empty(n_bootstrap, dtype=np.float64)

    for start in range(0, n_bootstrap, batch_size):
        stop = min(start + batch_size, n_bootstrap)
        current_batch_size = stop - start

        indices = rng.integers(0, n, size=(current_batch_size, n))
        bootstrap_means[start:stop] = differences[indices].mean(axis=1)

    ci_low, ci_high = np.quantile(
        bootstrap_means,
        [alpha / 2, 1 - alpha / 2]
    )

    return float(ci_low), float(ci_high)


## 3. Ordered vs. Shuffled — FEA

All cells in this section concern **Ordered vs. Shuffled** only. They construct the paired analysis table, verify accuracy, perform the primary test and confidence-interval calculation, and report sensitivity and participant-level results.

### 3.1 Ordered vs. Shuffled: Construct Reenactment-Level Analysis Table

For each reenactment, let $O_i$ indicate whether the Ordered FEA prediction is correct and $A_i$ indicate whether the Shuffled FEA prediction is correct.

The paired difference is

$d_i = O_i - A_i$

with support $\{-1, 0, 1\}$:

- $+1$: only Ordered FEA is correct
- $-1$: only Shuffled FEA is correct
- $0$: both models have the same correctness outcome

In [5]:
analysis_df = prediction_df[
    ["reenactment_id", "participant_id", "true_label_id", "ordered_pred_id", "shuffled_pred_id"]
].copy()

analysis_df["ordered_fea_correct"] = (
    analysis_df["ordered_pred_id"] == analysis_df["true_label_id"]
)
analysis_df["shuffled_fea_correct"] = (
    analysis_df["shuffled_pred_id"] == analysis_df["true_label_id"]
)
analysis_df["difference"] = (
    analysis_df["ordered_fea_correct"].astype(int)
    - analysis_df["shuffled_fea_correct"].astype(int)
)

assert len(analysis_df) == 378
assert set(analysis_df["difference"].unique()).issubset({-1, 0, 1})

analysis_df.head()

,reenactment_id,participant_id,true_label_id,ordered_pred_id,shuffled_pred_id,ordered_fea_correct,shuffled_fea_correct,difference
0,1700478995850-2-1-1-0-0,1,0,0,0,True,True,0
1,1700478998549-2-1-1-1-5,1,5,5,5,True,True,0
2,1700479001137-2-1-1-2-3,1,3,3,3,True,True,0
3,1700479004312-2-1-1-3-0,1,0,4,4,False,False,0
4,1700479005401-2-1-1-4-0,1,0,0,0,True,True,0


### 3.2 Ordered vs. Shuffled: Verify Reported Performance

In [6]:
ordered_fea_accuracy = analysis_df["ordered_fea_correct"].mean()
shuffled_fea_accuracy = analysis_df["shuffled_fea_correct"].mean()

ordered_correct = int(analysis_df["ordered_fea_correct"].sum())
shuffled_correct = int(analysis_df["shuffled_fea_correct"].sum())

assert ordered_correct == 296
assert shuffled_correct == 265

print(f"Ordered FEA:  {ordered_correct}/378 = {100 * ordered_fea_accuracy:.2f}%")
print(f"Shuffled FEA: {shuffled_correct}/378 = {100 * shuffled_fea_accuracy:.2f}%")
print(f"Ordered - Shuffled difference: {100 * (ordered_fea_accuracy - shuffled_fea_accuracy):.2f} percentage points")

Ordered FEA:  296/378 = 78.31%
Shuffled FEA: 265/378 = 70.11%
Ordered - Shuffled difference: 8.20 percentage points


### 3.3 Ordered vs. Shuffled: Primary Ordered FEA vs. Shuffled FEA Comparison

The primary estimand is

$\Delta = \mathrm{Accuracy}_{Ordered} - \mathrm{Accuracy}_{Shuffled}$.

The primary inferential test is an **exact two-sided McNemar test** over the 378 paired reenactments.

The $2\times2$ paired-correctness table is organized as:

| | Shuffled correct | Shuffled wrong |
|---|---:|---:|
| Ordered correct | both correct | Ordered only correct |
| Ordered wrong | Shuffled only correct | both wrong |

Only the two discordant cells contribute to McNemar's test. Positive values of $\Delta$ favor Ordered FEA.

In [7]:
ordered_correct_array = analysis_df["ordered_fea_correct"].to_numpy(dtype=bool)
shuffled_correct_array = analysis_df["shuffled_fea_correct"].to_numpy(dtype=bool)

both_correct = int(np.sum(ordered_correct_array & shuffled_correct_array))
ordered_only_correct = int(np.sum(ordered_correct_array & ~shuffled_correct_array))
shuffled_only_correct = int(np.sum(~ordered_correct_array & shuffled_correct_array))
both_wrong = int(np.sum(~ordered_correct_array & ~shuffled_correct_array))

mcnemar_table = np.array([
    [both_correct, ordered_only_correct],
    [shuffled_only_correct, both_wrong]
])

mcnemar_result = mcnemar(mcnemar_table, exact=True, correction=False)

print("Paired correctness table:")
print(mcnemar_table)
print()
print(f"Both correct:          {both_correct}")
print(f"Ordered only correct:  {ordered_only_correct}")
print(f"Shuffled only correct: {shuffled_only_correct}")
print(f"Both wrong:            {both_wrong}")
print()
print(f"Exact two-sided McNemar p-value: {mcnemar_result.pvalue:.12f}")

Paired correctness table:
[[255  41]
 [ 10  72]]

Both correct:          255
Ordered only correct:  41
Shuffled only correct: 10
Both wrong:            72

Exact two-sided McNemar p-value: 0.000014737716


### 3.4 Ordered vs. Shuffled: Paired Bootstrap Confidence Interval

McNemar's test provides the primary hypothesis test. To quantify the effect size and its uncertainty, a paired bootstrap is used for the accuracy difference $\Delta$.

The 378 reenactment-level pairs are resampled with replacement. Ordered and Shuffled correctness remain paired within each resampled reenactment.

The percentile interval below is a 95% bootstrap confidence interval for $\Delta$. No additional bootstrap hypothesis test is used because the exact McNemar test is the primary inferential test.

In [8]:
differences = analysis_df["difference"].to_numpy(dtype=float)

ci_low, ci_high = paired_bootstrap_ci(
    differences=differences,
    n_bootstrap=N_BOOTSTRAP,
    batch_size=BOOTSTRAP_BATCH_SIZE,
    alpha=ALPHA,
    random_seed=RANDOM_SEED
)

difference_mean = differences.mean()

primary_result_df = pd.DataFrame([{
    "comparison": "Ordered FEA vs. Shuffled FEA",
    "n_reenactments": len(analysis_df),
    "ordered_fea_accuracy": ordered_fea_accuracy,
    "shuffled_fea_accuracy": shuffled_fea_accuracy,
    "difference_ordered_minus_shuffled_pp": 100 * difference_mean,
    "ci_low_pp": 100 * ci_low,
    "ci_high_pp": 100 * ci_high,
    "both_correct": both_correct,
    "ordered_only_correct": ordered_only_correct,
    "shuffled_only_correct": shuffled_only_correct,
    "both_wrong": both_wrong,
    "mcnemar_statistic": float(mcnemar_result.statistic),
    "p_value": float(mcnemar_result.pvalue)
}])

row = primary_result_df.iloc[0]

print(f"Ordered - Shuffled FEA accuracy difference: {row['difference_ordered_minus_shuffled_pp']:.2f} percentage points")
print(f"{100 * (1 - ALPHA):.0f}% paired-bootstrap CI: [{row['ci_low_pp']:.2f}, {row['ci_high_pp']:.2f}] percentage points")
print(f"Exact two-sided McNemar p-value: {row['p_value']:.12f}")

primary_result_df

Ordered - Shuffled FEA accuracy difference: 8.20 percentage points
95% paired-bootstrap CI: [4.76, 11.90] percentage points
Exact two-sided McNemar p-value: 0.000014737716


,comparison,n_reenactments,ordered_fea_accuracy,shuffled_fea_accuracy,difference_ordered_minus_shuffled_pp,ci_low_pp,ci_high_pp,both_correct,ordered_only_correct,shuffled_only_correct,both_wrong,mcnemar_statistic,p_value
0,Ordered FEA vs. Shuffled FEA,378,0.783069,0.701058,8.201058,4.761905,11.904762,255,41,10,72,10.0,0.000015


### 3.5 Ordered vs. Shuffled: Sensitivity Check

As a sensitivity analysis, the same 378 paired reenactment-level differences $d_i = O_i - A_i$ are tested with a one-sample t-test against a mean of zero.

This is not a separate primary hypothesis test. It only checks whether a conventional parametric test leads to the same substantive conclusion as the exact McNemar analysis.

In [9]:
sensitivity_test = ttest_1samp(differences, popmean=0.0)

sensitivity_result_df = pd.DataFrame([{
    "comparison": "Ordered FEA vs. Shuffled FEA",
    "n_reenactments": len(differences),
    "mean_difference_pp": 100 * differences.mean(),
    "t_statistic": float(sensitivity_test.statistic),
    "df": int(sensitivity_test.df),
    "p_value": float(sensitivity_test.pvalue)
}])

row = sensitivity_result_df.iloc[0]

print(f"Mean difference: {row['mean_difference_pp']:.2f} percentage points")
print(f"t({int(row['df'])}) = {row['t_statistic']:.3f}")
print(f"Two-sided sensitivity-check p-value: {row['p_value']:.12f}")

sensitivity_result_df

Mean difference: 8.20 percentage points
t(377) = 4.447
Two-sided sensitivity-check p-value: 0.000011452404


,comparison,n_reenactments,mean_difference_pp,t_statistic,df,p_value
0,Ordered FEA vs. Shuffled FEA,378,8.201058,4.44739,377,0.000011


### 3.6 Ordered vs. Shuffled: Participant-Level Heterogeneity Check

This descriptive check examines whether the Ordered-vs.-Shuffled FEA difference is directionally consistent across the eight held-out participants or is mainly driven by a small number of participants.

For each participant, the table reports the number of reenactments, Ordered FEA accuracy, Shuffled FEA accuracy, and the difference $\mathrm{Accuracy}_{Ordered} - \mathrm{Accuracy}_{Shuffled}$.

No participant-level significance test or multiplicity correction is applied. The primary inference remains the paired reenactment-level McNemar analysis above.

In [10]:
participant_result_df = (
    analysis_df
    .groupby("participant_id", as_index=False)
    .agg(
        n_reenactments=("reenactment_id", "size"),
        ordered_fea_accuracy=("ordered_fea_correct", "mean"),
        shuffled_fea_accuracy=("shuffled_fea_correct", "mean")
    )
)

participant_result_df["difference_pp"] = 100 * (
    participant_result_df["ordered_fea_accuracy"]
    - participant_result_df["shuffled_fea_accuracy"]
)

n_ordered_better = int((participant_result_df["difference_pp"] > 0).sum())
n_shuffled_better = int((participant_result_df["difference_pp"] < 0).sum())
n_equal = int((participant_result_df["difference_pp"] == 0).sum())
median_participant_difference_pp = participant_result_df["difference_pp"].median()

print(f"Participants favoring Ordered FEA: {n_ordered_better}/8")
print(f"Participants favoring Shuffled FEA: {n_shuffled_better}/8")
print(f"Participants tied:                  {n_equal}/8")
print(f"Median participant difference (Ordered - Shuffled FEA): {median_participant_difference_pp:.2f} percentage points")

participant_result_df

Participants favoring Ordered FEA: 7/8
Participants favoring Shuffled FEA: 1/8
Participants tied:                  0/8
Median participant difference (Ordered - Shuffled FEA): 5.61 percentage points


,participant_id,n_reenactments,ordered_fea_accuracy,shuffled_fea_accuracy,difference_pp
0,1,47,0.723404,0.659574,6.382979
1,8,54,0.722222,0.666667,5.555556
2,10,46,0.804348,0.586957,21.739130
3,13,46,0.934783,0.717391,21.739130
4,15,48,0.625000,0.645833,-2.083333
5,18,43,0.976744,0.953488,2.325581
6,23,53,0.792453,0.735849,5.660377
7,27,41,0.707317,0.658537,4.878049


### 3.7 Ordered vs. Shuffled: summary and retained results

Retain this comparison’s tables for the joint correction and exports; no test is recalculated.

In [11]:
summary_df = pd.DataFrame({
    "metric": [
        "Ordered FEA accuracy",
        "Shuffled FEA accuracy",
        "Ordered - Shuffled FEA difference (pp)",
        "Paired-bootstrap CI low (pp)",
        "Paired-bootstrap CI high (pp)",
        "Exact McNemar p-value",
        "Ordered-only correct reenactments",
        "Shuffled-only correct reenactments",
        "Sensitivity t statistic",
        "Sensitivity t-test p-value",
        "Participants favoring Ordered FEA",
        "Participants favoring Shuffled FEA",
        "Participants tied",
        "Median participant difference Ordered - Shuffled FEA (pp)"
    ],
    "value": [
        ordered_fea_accuracy,
        shuffled_fea_accuracy,
        100 * difference_mean,
        100 * ci_low,
        100 * ci_high,
        mcnemar_result.pvalue,
        ordered_only_correct,
        shuffled_only_correct,
        sensitivity_test.statistic,
        sensitivity_test.pvalue,
        n_ordered_better,
        n_shuffled_better,
        n_equal,
        median_participant_difference_pp
    ]
})

summary_df

,metric,value
0,Ordered FEA accuracy,0.783069
1,Shuffled FEA accuracy,0.701058
2,Ordered - Shuffled FEA difference (pp),8.201058
3,Paired-bootstrap CI low (pp),4.761905
4,Paired-bootstrap CI high (pp),11.904762
5,Exact McNemar p-value,0.000015
6,Ordered-only correct reenactments,41.000000
7,Shuffled-only correct reenactments,10.000000
8,Sensitivity t statistic,4.447390
9,Sensitivity t-test p-value,0.000011


In [12]:
results["shuffled"] = {
    "reenactment_table": analysis_df.copy(),
    "primary_result": primary_result_df.copy(),
    "sensitivity_ttest": sensitivity_result_df.copy(),
    "participant_level_results": participant_result_df.copy(),
    "summary": summary_df.copy(),
}

## 4. Ordered vs. Mean — FEA

All cells in this section concern **Ordered vs. Mean** only. They construct the paired analysis table, verify accuracy, perform the primary test and confidence-interval calculation, and report sensitivity and participant-level results.

### 4.1 Ordered vs. Mean: Construct Reenactment-Level Analysis Table

For each reenactment, let $O_i$ indicate whether the Ordered FEA prediction is correct and $A_i$ indicate whether the Mean FEA prediction is correct.

The paired difference is

$d_i = O_i - A_i$

with support $\{-1, 0, 1\}$:

- $+1$: only Ordered FEA is correct
- $-1$: only Mean FEA is correct
- $0$: both models have the same correctness outcome

In [13]:
analysis_df = prediction_df[
    ["reenactment_id", "participant_id", "true_label_id", "ordered_pred_id", "mean_pred_id"]
].copy()

analysis_df["ordered_fea_correct"] = (
    analysis_df["ordered_pred_id"] == analysis_df["true_label_id"]
)
analysis_df["mean_fea_correct"] = (
    analysis_df["mean_pred_id"] == analysis_df["true_label_id"]
)
analysis_df["difference"] = (
    analysis_df["ordered_fea_correct"].astype(int)
    - analysis_df["mean_fea_correct"].astype(int)
)

assert len(analysis_df) == 378
assert set(analysis_df["difference"].unique()).issubset({-1, 0, 1})

analysis_df.head()

,reenactment_id,participant_id,true_label_id,ordered_pred_id,mean_pred_id,ordered_fea_correct,mean_fea_correct,difference
0,1700478995850-2-1-1-0-0,1,0,0,0,True,True,0
1,1700478998549-2-1-1-1-5,1,5,5,5,True,True,0
2,1700479001137-2-1-1-2-3,1,3,3,3,True,True,0
3,1700479004312-2-1-1-3-0,1,0,4,4,False,False,0
4,1700479005401-2-1-1-4-0,1,0,0,0,True,True,0


### 4.2 Ordered vs. Mean: Verify Reported Performance

In [14]:
ordered_fea_accuracy = analysis_df["ordered_fea_correct"].mean()
mean_fea_accuracy = analysis_df["mean_fea_correct"].mean()

ordered_correct = int(analysis_df["ordered_fea_correct"].sum())
mean_correct = int(analysis_df["mean_fea_correct"].sum())

assert ordered_correct == 296
assert mean_correct == 261

print(f"Ordered FEA:  {ordered_correct}/378 = {100 * ordered_fea_accuracy:.2f}%")
print(f"Mean FEA: {mean_correct}/378 = {100 * mean_fea_accuracy:.2f}%")
print(f"Ordered - Mean difference: {100 * (ordered_fea_accuracy - mean_fea_accuracy):.2f} percentage points")

Ordered FEA:  296/378 = 78.31%
Mean FEA: 261/378 = 69.05%
Ordered - Mean difference: 9.26 percentage points


### 4.3 Ordered vs. Mean: Primary Ordered FEA vs. Mean FEA Comparison

The primary estimand is

$\Delta = \mathrm{Accuracy}_{Ordered} - \mathrm{Accuracy}_{Mean}$.

The primary inferential test is an **exact two-sided McNemar test** over the 378 paired reenactments.

The $2\times2$ paired-correctness table is organized as:

| | Mean correct | Mean wrong |
|---|---:|---:|
| Ordered correct | both correct | Ordered only correct |
| Ordered wrong | Mean only correct | both wrong |

Only the two discordant cells contribute to McNemar's test. Positive values of $\Delta$ favor Ordered FEA.

In [15]:
ordered_correct_array = analysis_df["ordered_fea_correct"].to_numpy(dtype=bool)
mean_correct_array = analysis_df["mean_fea_correct"].to_numpy(dtype=bool)

both_correct = int(np.sum(ordered_correct_array & mean_correct_array))
ordered_only_correct = int(np.sum(ordered_correct_array & ~mean_correct_array))
mean_only_correct = int(np.sum(~ordered_correct_array & mean_correct_array))
both_wrong = int(np.sum(~ordered_correct_array & ~mean_correct_array))

mcnemar_table = np.array([
    [both_correct, ordered_only_correct],
    [mean_only_correct, both_wrong]
])

mcnemar_result = mcnemar(mcnemar_table, exact=True, correction=False)

print("Paired correctness table:")
print(mcnemar_table)
print()
print(f"Both correct:          {both_correct}")
print(f"Ordered only correct:  {ordered_only_correct}")
print(f"Mean only correct: {mean_only_correct}")
print(f"Both wrong:            {both_wrong}")
print()
print(f"Exact two-sided McNemar p-value: {mcnemar_result.pvalue:.12f}")

Paired correctness table:
[[254  42]
 [  7  75]]

Both correct:          254
Ordered only correct:  42
Mean only correct: 7
Both wrong:            75

Exact two-sided McNemar p-value: 0.000000362458


### 4.4 Ordered vs. Mean: Paired Bootstrap Confidence Interval

McNemar's test provides the primary hypothesis test. To quantify the effect size and its uncertainty, a paired bootstrap is used for the accuracy difference $\Delta$.

The 378 reenactment-level pairs are resampled with replacement. Ordered and Mean correctness remain paired within each resampled reenactment.

The percentile interval below is a 95% bootstrap confidence interval for $\Delta$. No additional bootstrap hypothesis test is used because the exact McNemar test is the primary inferential test.

In [16]:
differences = analysis_df["difference"].to_numpy(dtype=float)

ci_low, ci_high = paired_bootstrap_ci(
    differences=differences,
    n_bootstrap=N_BOOTSTRAP,
    batch_size=BOOTSTRAP_BATCH_SIZE,
    alpha=ALPHA,
    random_seed=RANDOM_SEED
)

difference_mean = differences.mean()

primary_result_df = pd.DataFrame([{
    "comparison": "Ordered FEA vs. Mean FEA",
    "n_reenactments": len(analysis_df),
    "ordered_fea_accuracy": ordered_fea_accuracy,
    "mean_fea_accuracy": mean_fea_accuracy,
    "difference_ordered_minus_mean_pp": 100 * difference_mean,
    "ci_low_pp": 100 * ci_low,
    "ci_high_pp": 100 * ci_high,
    "both_correct": both_correct,
    "ordered_only_correct": ordered_only_correct,
    "mean_only_correct": mean_only_correct,
    "both_wrong": both_wrong,
    "mcnemar_statistic": float(mcnemar_result.statistic),
    "p_value": float(mcnemar_result.pvalue)
}])

row = primary_result_df.iloc[0]

print(f"Ordered - Mean FEA accuracy difference: {row['difference_ordered_minus_mean_pp']:.2f} percentage points")
print(f"{100 * (1 - ALPHA):.0f}% paired-bootstrap CI: [{row['ci_low_pp']:.2f}, {row['ci_high_pp']:.2f}] percentage points")
print(f"Exact two-sided McNemar p-value: {row['p_value']:.12f}")

primary_result_df

Ordered - Mean FEA accuracy difference: 9.26 percentage points
95% paired-bootstrap CI: [5.82, 12.96] percentage points
Exact two-sided McNemar p-value: 0.000000362458


,comparison,n_reenactments,ordered_fea_accuracy,mean_fea_accuracy,difference_ordered_minus_mean_pp,ci_low_pp,ci_high_pp,both_correct,ordered_only_correct,mean_only_correct,both_wrong,mcnemar_statistic,p_value
0,Ordered FEA vs. Mean FEA,378,0.783069,0.690476,9.259259,5.820106,12.962963,254,42,7,75,7.0,3.624578e-07


### 4.5 Ordered vs. Mean: Sensitivity Check

As a sensitivity analysis, the same 378 paired reenactment-level differences $d_i = O_i - A_i$ are tested with a one-sample t-test against a mean of zero.

This is not a separate primary hypothesis test. It only checks whether a conventional parametric test leads to the same substantive conclusion as the exact McNemar analysis.

In [17]:
sensitivity_test = ttest_1samp(differences, popmean=0.0)

sensitivity_result_df = pd.DataFrame([{
    "comparison": "Ordered FEA vs. Mean FEA",
    "n_reenactments": len(differences),
    "mean_difference_pp": 100 * differences.mean(),
    "t_statistic": float(sensitivity_test.statistic),
    "df": int(sensitivity_test.df),
    "p_value": float(sensitivity_test.pvalue)
}])

row = sensitivity_result_df.iloc[0]

print(f"Mean difference: {row['mean_difference_pp']:.2f} percentage points")
print(f"t({int(row['df'])}) = {row['t_statistic']:.3f}")
print(f"Two-sided sensitivity-check p-value: {row['p_value']:.12f}")

sensitivity_result_df

Mean difference: 9.26 percentage points
t(377) = 5.167
Two-sided sensitivity-check p-value: 0.000000386086


,comparison,n_reenactments,mean_difference_pp,t_statistic,df,p_value
0,Ordered FEA vs. Mean FEA,378,9.259259,5.167177,377,3.860863e-07


### 4.6 Ordered vs. Mean: Participant-Level Heterogeneity Check

This descriptive check examines whether the Ordered-vs.-Mean FEA difference is directionally consistent across the eight held-out participants or is mainly driven by a small number of participants.

For each participant, the table reports the number of reenactments, Ordered FEA accuracy, Mean FEA accuracy, and the difference $\mathrm{Accuracy}_{Ordered} - \mathrm{Accuracy}_{Mean}$.

No participant-level significance test or multiplicity correction is applied. The primary inference remains the paired reenactment-level McNemar analysis above.

In [18]:
participant_result_df = (
    analysis_df
    .groupby("participant_id", as_index=False)
    .agg(
        n_reenactments=("reenactment_id", "size"),
        ordered_fea_accuracy=("ordered_fea_correct", "mean"),
        mean_fea_accuracy=("mean_fea_correct", "mean")
    )
)

participant_result_df["difference_pp"] = 100 * (
    participant_result_df["ordered_fea_accuracy"]
    - participant_result_df["mean_fea_accuracy"]
)

n_ordered_better = int((participant_result_df["difference_pp"] > 0).sum())
n_mean_better = int((participant_result_df["difference_pp"] < 0).sum())
n_equal = int((participant_result_df["difference_pp"] == 0).sum())
median_participant_difference_pp = participant_result_df["difference_pp"].median()

print(f"Participants favoring Ordered FEA: {n_ordered_better}/8")
print(f"Participants favoring Mean FEA: {n_mean_better}/8")
print(f"Participants tied:                  {n_equal}/8")
print(f"Median participant difference (Ordered - Mean FEA): {median_participant_difference_pp:.2f} percentage points")

participant_result_df

Participants favoring Ordered FEA: 6/8
Participants favoring Mean FEA: 0/8
Participants tied:                  2/8
Median participant difference (Ordered - Mean FEA): 7.84 percentage points


,participant_id,n_reenactments,ordered_fea_accuracy,mean_fea_accuracy,difference_pp
0,1,47,0.723404,0.680851,4.255319
1,8,54,0.722222,0.574074,14.814815
2,10,46,0.804348,0.630435,17.391304
3,13,46,0.934783,0.739130,19.565217
4,15,48,0.625000,0.562500,6.250000
5,18,43,0.976744,0.976744,0.000000
6,23,53,0.792453,0.698113,9.433962
7,27,41,0.707317,0.707317,0.000000


### 4.7 Ordered vs. Mean: summary and retained results

Retain this comparison’s tables for the joint correction and exports; no test is recalculated.

In [19]:
summary_df = pd.DataFrame({
    "metric": [
        "Ordered FEA accuracy",
        "Mean FEA accuracy",
        "Ordered - Mean FEA difference (pp)",
        "Paired-bootstrap CI low (pp)",
        "Paired-bootstrap CI high (pp)",
        "Exact McNemar p-value",
        "Ordered-only correct reenactments",
        "Mean-only correct reenactments",
        "Sensitivity t statistic",
        "Sensitivity t-test p-value",
        "Participants favoring Ordered FEA",
        "Participants favoring Mean FEA",
        "Participants tied",
        "Median participant difference Ordered - Mean FEA (pp)"
    ],
    "value": [
        ordered_fea_accuracy,
        mean_fea_accuracy,
        100 * difference_mean,
        100 * ci_low,
        100 * ci_high,
        mcnemar_result.pvalue,
        ordered_only_correct,
        mean_only_correct,
        sensitivity_test.statistic,
        sensitivity_test.pvalue,
        n_ordered_better,
        n_mean_better,
        n_equal,
        median_participant_difference_pp
    ]
})

summary_df

,metric,value
0,Ordered FEA accuracy,7.830688e-01
1,Mean FEA accuracy,6.904762e-01
2,Ordered - Mean FEA difference (pp),9.259259e+00
3,Paired-bootstrap CI low (pp),5.820106e+00
4,Paired-bootstrap CI high (pp),1.296296e+01
5,Exact McNemar p-value,3.624578e-07
6,Ordered-only correct reenactments,4.200000e+01
7,Mean-only correct reenactments,7.000000e+00
8,Sensitivity t statistic,5.167177e+00
9,Sensitivity t-test p-value,3.860863e-07


In [20]:
results["mean"] = {
    "reenactment_table": analysis_df.copy(),
    "primary_result": primary_result_df.copy(),
    "sensitivity_ttest": sensitivity_result_df.copy(),
    "participant_level_results": participant_result_df.copy(),
    "summary": summary_df.copy(),
}

## 5. Holm correction — both FEA primary tests

The family contains **Ordered vs. Shuffled** and **Ordered vs. Mean**. Use the two primary p-values already calculated above. Holm controls family-wise error at 0.05 within this modality. Sensitivity tests and participant summaries are excluded; confidence intervals remain unchanged.

The inputs are the two exact McNemar p-values.

In [21]:
holm_family_df = pd.DataFrame({
    "alternative": ["shuffled", "mean"],
    "p_value": [results[a]["primary_result"].loc[0, "p_value"] for a in ["shuffled", "mean"]],
})
reject, p_holm, _, _ = multipletests(holm_family_df["p_value"], alpha=ALPHA, method="holm")
holm_family_df["p_value_holm"] = p_holm
holm_family_df["reject_holm"] = reject
holm_family_df

,alternative,p_value,p_value_holm,reject_holm
0,shuffled,1.473772e-05,1.473772e-05,True
1,mean,3.624578e-07,7.249157e-07,True


## 6. Export both comparisons

Keep the original five result-CSV filenames per comparison. Add raw and adjusted p-values to the primary results and adjusted values to the summaries. Export one common Holm-family table for this modality.

In [22]:
OUTPUT_DIR = Path("statistical_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for row in holm_family_df.itertuples(index=False):
    tables = results[row.alternative]
    primary = tables["primary_result"]
    primary["p_value_holm"] = row.p_value_holm
    primary["reject_holm"] = row.reject_holm
    primary["holm_family"] = "fea_sequence_order"
    primary["holm_family_size"] = 2
    primary["alpha"] = ALPHA
    summary = pd.concat([tables["summary"], pd.DataFrame({
        "metric": ["Primary Holm-adjusted p-value (within modality)", "Reject primary null after Holm correction", "Holm family size", "Family-wise alpha"],
        "value": [row.p_value_holm, int(row.reject_holm), 2, ALPHA],
    })], ignore_index=True)
    for name, table in tables.items():
        if name == "summary":
            table = summary
        table.to_csv(OUTPUT_DIR / f"ordered_fea_vs_{row.alternative}_fea_{name}.csv",
                     index=False and name == "reenactment_table")

holm_family_df.to_csv(OUTPUT_DIR / "fea_sequence_order_holm_family.csv", index=False)
print(f"Results written to: {OUTPUT_DIR.resolve()}")

Results written to: /workspace/repos/emohevrdb-dfer/5_dynamic_facial_expression_recognition/5_2_fea_sequence_based_fer/5_2_4_sequence_order_ablation/statistical-significance-tests/statistical_results
